## Setup & Initialization

In [1]:
import random
import pyspark
import os
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

from pyspark.sql.functions import *

In [2]:
reload(cc)

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/s

In [3]:
spark = cc.startLocalCluster("factTreasureHuntIncremental")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


## Load dimension tables to get surrogate keys

In [4]:
dim_user = spark.read.format("delta").load("spark-warehouse/dim_user")
dim_treasure = spark.read.format("delta").load("spark-warehouse/dim_treasure")
dim_date = spark.read.format("delta").load("spark-warehouse/dim_date")
dim_season = spark.read.format("delta").load("spark-warehouse/dim_season")
dim_weather = spark.read.format("delta").load("spark-warehouse/dim_weather")
dim_hour = spark.read.format("delta").load("spark-warehouse/dim_hour")

In [6]:
dim_season.show()

+---------+-----------+
|season_SK|season_name|
+---------+-----------+
|        1|     Winter|
|        2|     Spring|
|        3|     Summer|
|        4|     Autumn|
+---------+-----------+



In [48]:
dim_weather.show()

+----------+---------------+
|weather_sk|           name|
+----------+---------------+
|         1|           Rain|
|         2|        No Rain|
|         3|Weather Unknown|
+----------+---------------+



# Reading the source table

In [5]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


In [6]:
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time") #\
                                # .filter(F.col("log_time").isNotNull())
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_treasure_stages = cc.read_postgres_without_partition(spark, "treasure_stages")

print('user size: ', df_operational_user.count())  ## 486523
print('treasure size: ', df_operational_treasure.count())  ## 128294
print('treasure_log size: ', df_operational_treasure_log.count())  ## 1789438 --it will be one more
print('city size: ', df_operational_city.count())  ## 1264588
print('treasure_stages size: ', df_operational_treasure_stages.count())  ## 504119

user size:  486523
treasure size:  128294
treasure_log size:  1789439
city size:  1264588
treasure_stages size:  504119


### Selecting and renaming columns for easier joins and transformations
also, we can filter or aggregate here if needed to optimize later steps

In [17]:
df_operational_city = df_operational_city.select(
    "city_id",
    "postal_code",
)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)

df_operational_treasure_log = (
    df_operational_treasure_log
    .filter(F.col("log_time").isNotNull())
    .withColumn("duration_minutes",
                (F.unix_timestamp("log_time") - F.unix_timestamp("session_start")) / 60
                )
    .withColumn(
        "hour",
        F.hour(F.date_trunc("hour", F.col("log_time") + F.expr("INTERVAL 30 MINUTES")))
        #todo round to nearest hour
        # F.hour("log_time")
    )
    .withColumn("full_date", F.to_date("log_time"))
    .select(
        F.col("hunter_id").alias("user_id"),
        "duration_minutes",
        "treasure_id",
        "hour",
        "full_date"
    )
)

df_operational_treasure = df_operational_treasure.select(
    F.col("id").alias("treasure_id"),
    "difficulty",
    "terrain"
)

df_operational_treasure_stages = df_operational_treasure_stages \
    .groupBy("treasure_id") \
    .agg(F.count("stages_id").alias("stage_count"))

print('city size: ', df_operational_city.count())  ## 1264588
print('user size: ', df_operational_user.count())  ## 486523
print('treasure_log size: ', df_operational_treasure_log.count())  ## 1789438 --it will be one more
print('treasure size: ', df_operational_treasure.count())  ## 128294

city size:  1264588
user size:  486523
treasure_log size:  1789439
treasure size:  128294


In [18]:
## Joining the stage count back to the treasure table, filling missing stage counts with 0
df_operational_treasure = (
    df_operational_treasure
    .join(df_operational_treasure_stages, "treasure_id", "left")
    .select(
        "treasure_id",
        "difficulty",
        "terrain",
        F.coalesce(F.col("stage_count"), F.lit(0)).cast("integer").alias("stage_count")
    )
)

df_operational_treasure.show(10, truncate=False)

+-------------------------------------------------+----------+-------+-----------+
|treasure_id                                      |difficulty|terrain|stage_count|
+-------------------------------------------------+----------+-------+-----------+
|[00 03 72 3C 7C C3 42 62 A6 F3 E8 A3 E7 B8 05 63]|3         |2      |7          |
|[00 04 62 1B 29 E8 48 DB 8D 98 C6 95 47 E5 1D A0]|1         |2      |4          |
|[00 06 05 10 FC 0C 4E 85 A3 BD 1C 99 9F B6 60 13]|2         |2      |9          |
|[00 08 A9 BF BE 4B 4F 22 86 8B 2A 56 E5 BD B0 54]|3         |2      |1          |
|[00 08 B9 20 1B F9 4A 0F A6 F3 0B 56 AB 1C FF E0]|1         |2      |1          |
|[00 05 1E A2 05 87 47 1F 80 A3 28 81 99 C8 21 D3]|0         |2      |1          |
|[00 08 B8 9F B1 D5 46 0F 94 7A 6A 24 C7 65 4F 53]|3         |0      |6          |
|[00 08 E2 70 CB AD 4E 39 8A 2F F2 23 F0 32 A1 C8]|2         |2      |3          |
|[00 00 3E 2C B1 40 41 B1 97 F0 00 33 C0 97 E0 70]|3         |1      |4          |
|[00

In [53]:
df_operational_treasure_log.show(10, truncate=False)

+-------------------------------------------------+----------------+-------------------------------------------------+----+----------+
|user_id                                          |duration_minutes|treasure_id                                      |hour|full_date |
+-------------------------------------------------+----------------+-------------------------------------------------+----+----------+
|[2E DF B2 3A 68 19 49 51 AE 9A 4F FB 9C 0F 2A 76]|78.0            |[9F 81 9C 74 88 44 4B 06 A9 27 11 6E BF EC 4A 68]|2   |2020-10-19|
|[07 FF F6 69 FC AB 4A CC 89 DB CC 05 3A 84 E7 4A]|54.0            |[8D 1E E5 DE E2 54 45 E5 8E 19 4B 3C FE 71 D4 5C]|9   |2020-11-29|
|[F6 29 EF 08 13 D3 4A F5 B9 E0 C8 FD 18 93 0D 82]|21.0            |[7A 37 97 9A 05 C5 40 D8 9B 87 5C 59 57 65 D3 2F]|7   |2020-10-19|
|[4B 88 92 E3 E1 B1 45 28 B2 9A 64 0A 7C B7 3D 49]|11.0            |[8F 50 90 BF 2B 50 4C AF 9F 6B FD CB DE 1B B3 71]|18  |2020-10-10|
|[B4 37 A7 24 47 F1 49 68 95 7F C7 50 EB C8 D6 CF]|226.

# Extracting weather information the json files

In [19]:
# Load weather JSON files
weather_json_path = "./data/weather_data/*.json"
weather_json = spark.read.option("multiLine", True).json(weather_json_path)

In [11]:
# This returns a Python list of the column names
print(weather_json.columns)

['base', 'clouds', 'cod', 'coord', 'dt', 'id', 'main', 'name', 'rain', 'sys', 'timezone', 'visibility', 'weather', 'wind', 'zipCode']


In [12]:
# Explode the weather array to get individual weather entries
weather_json.show(10, truncate=False)

+--------+------+---+---------------------+----------+-------+-----------------------------------------------------+------------+------+----------------------------------------+--------+----------+--------------------------------------------------------+-----------------+-------+
|base    |clouds|cod|coord                |dt        |id     |main                                                 |name        |rain  |sys                                     |timezone|visibility|weather                                                 |wind             |zipCode|
+--------+------+---+---------------------+----------+-------+-----------------------------------------------------+------------+------+----------------------------------------+--------+----------+--------------------------------------------------------+-----------------+-------+
|stations|{100} |200|{74.4435, 19.0063}   |1602326355|3163858|{298.74, 933, 64, 1015, 1015, 298.48, 300.05, 297.56}|Bjørnøya    |{3.16}|{IT, 2075663, 1602304

In [20]:
weather_json.count() #243

243

In [21]:
weather_df = (
    weather_json
    .withColumn("weather_entry", F.explode("weather"))
    .withColumn("ts", F.to_timestamp(F.from_unixtime("dt")))
    .select(
        F.col("zipCode").alias("postal_code"),
        "ts",
        F.col("weather_entry.id").alias("weather_id")
    )
    .withColumn(
        "hour",
        F.hour(F.date_trunc("hour", F.col("ts") + F.expr("INTERVAL 30 MINUTES")))
        #todo round to nearest hour
        # F.hour("ts")
    )
    .withColumn(
        "full_date",
        F.to_date("ts")
    )
    .drop("ts")
)

weather_df.show(10, truncate=False)

+-----------+----------+----+----------+
|postal_code|weather_id|hour|full_date |
+-----------+----------+----+----------+
|9176       |200       |11  |2020-10-10|
|97500      |200       |13  |2023-03-12|
|97500      |200       |15  |2020-11-19|
|97500      |200       |17  |2021-07-24|
|98600      |200       |9   |2022-08-12|
|98600      |200       |20  |2023-02-05|
|9170       |200       |16  |2021-10-31|
|9170       |200       |13  |2022-10-30|
|9170       |200       |17  |2023-08-08|
|9171       |200       |22  |2023-04-12|
+-----------+----------+----+----------+
only showing top 10 rows


In [22]:
df_fact_raw = (
    df_operational_treasure_log.alias("log")
    .join(df_operational_user.alias("u"), "user_id", "left")
    .join(df_operational_city.alias("c"), "city_id", "left")
    .join(df_operational_treasure.alias("t"), "treasure_id", "left")
    .join(dim_user.alias("du").select("user_id", "user_sk"), "user_id", "left")
    .join(dim_treasure.alias("dt"), ["difficulty", "terrain", "stage_count"], "left")
    .join(dim_date.alias("dd"), "full_date", "left")
    .join(dim_hour.alias("dh"), "hour", "left")
    .join(weather_df.alias("w"), ["postal_code", "hour", "full_date"], "left")
)

df_fact_raw.show(10)

+-----------+----+----------+----------+-------+-----------+--------------------+--------------------+--------------------+----------------+-------+-----------+-------+---+-----+----+----------+------------+----------+-------+----------+
|postal_code|hour| full_date|difficulty|terrain|stage_count|             user_id|         treasure_id|             city_id|duration_minutes|user_sk|treasure_sk|date_sk|day|month|year|month_name|weekday_name|is_weekend|hour_sk|weather_id|
+-----------+----+----------+----------+-------+-----------+--------------------+--------------------+--------------------+----------------+-------+-----------+-------+---+-----+----+----------+------------+----------+-------+----------+
|      29531|   4|2023-07-06|         1|      3|          7|[69 62 66 5B 6B 4...|[38 1E 21 BC FA A...|[BC E7 E3 86 9D 1...|           137.0| 201564|        109|   1028|  6|    7|2023|      July|    Thursday|     false|      4|      NULL|
|      47624|  22|2023-05-03|         1|      2|

### Join to weather dimension using the weather_id -> weather_sk mapping logic

In [25]:
df_fact_raw = (
    df_fact_raw
    .withColumn(
        "weather_name",
        F.when((F.col("weather_id") >= 200) & (F.col("weather_id") <= 699), "Rain")
        .when(F.col("weather_id").isNotNull(), "No Rain")
        .otherwise("Weather Unknown")
    )
)

In [26]:
df_fact_raw = (
    df_fact_raw
    .join(dim_weather.alias("dw"), df_fact_raw.weather_name == F.col("dw.name"), "left")
)

df_fact_raw = df_fact_raw.drop("weather_name")

df_fact_raw.show(10, truncate=False)

+-----------+----+----------+----------+-------+-----------+-------------------------------------------------+-------------------------------------------------+-------------------------------------------------+----------------+-------+-----------+-------+---+-----+----+----------+------------+----------+-------+----------+----------+---------------+
|postal_code|hour|full_date |difficulty|terrain|stage_count|user_id                                          |treasure_id                                      |city_id                                          |duration_minutes|user_sk|treasure_sk|date_sk|day|month|year|month_name|weekday_name|is_weekend|hour_sk|weather_id|weather_sk|name           |
+-----------+----+----------+----------+-------+-----------+-------------------------------------------------+-------------------------------------------------+-------------------------------------------------+----------------+-------+-----------+-------+---+-----+----+----------+------------+--

In [27]:
df_weather_counts = (
    df_fact_raw
    .groupBy("weather_sk")
    .agg(F.count("*").alias("count"))
    .orderBy(F.col("count").desc())
)

df_weather_counts.show()

+----------+-------+
|weather_sk|  count|
+----------+-------+
|         3|1789431|
|         1|      7|
|         2|      1|
+----------+-------+



### Join to season dimension using the month -> season_sk mapping logic (based on standard meteorological seasons)

In [28]:
df_fact_raw = (
    df_fact_raw
    .withColumn(
        "season_name",
        F.when(F.month("full_date").isin(12, 1, 2), "Winter")
        .when(F.month("full_date").isin(3, 4, 5), "Spring")
        .when(F.month("full_date").isin(6, 7, 8), "Summer")
        .otherwise("Autumn")
    )
)

In [29]:
df_fact_raw = (
    df_fact_raw
    .join(dim_season.alias("ds"), "season_name", "left")
)

df_fact_raw.show(10, truncate=False)

+-----------+-----------+----+----------+----------+-------+-----------+-------------------------------------------------+-------------------------------------------------+-------------------------------------------------+----------------+-------+-----------+-------+---+-----+----+----------+------------+----------+-------+----------+----------+---------------+---------+
|season_name|postal_code|hour|full_date |difficulty|terrain|stage_count|user_id                                          |treasure_id                                      |city_id                                          |duration_minutes|user_sk|treasure_sk|date_sk|day|month|year|month_name|weekday_name|is_weekend|hour_sk|weather_id|weather_sk|name           |season_SK|
+-----------+-----------+----+----------+----------+-------+-----------+-------------------------------------------------+-------------------------------------------------+-------------------------------------------------+----------------+-------+-----

In [30]:
df_season_counts = (
    df_fact_raw
    .groupBy("season_sk")
    .agg(F.count("*").alias("count"))
    .orderBy(F.col("count").desc())
)

df_season_counts.show()

+---------+------+
|season_sk| count|
+---------+------+
|        3|451153|
|        2|448455|
|        4|446865|
|        1|442966|
+---------+------+



In [31]:
from pyspark.sql.types import LongType, IntegerType

df_fact_final = (
    df_fact_raw
    .select(
        F.col("user_sk").cast(LongType()),
        F.col("treasure_sk").cast(IntegerType()),
        F.col("date_sk").cast(LongType()),
        F.col("season_sk").cast(IntegerType()),
        F.col("hour_sk").cast(IntegerType()),
        F.col("weather_sk").cast(LongType()),
        F.col("duration_minutes").cast(IntegerType())
    )
    .fillna(0)
)
df_fact_final.show(10, truncate=False)

+-------+-----------+-------+---------+-------+----------+----------------+
|user_sk|treasure_sk|date_sk|season_sk|hour_sk|weather_sk|duration_minutes|
+-------+-----------+-------+---------+-------+----------+----------------+
|479279 |269        |536    |2        |15     |3         |147             |
|68988  |200        |465    |1        |8      |3         |149             |
|42640  |0          |957    |2        |8      |3         |65              |
|299560 |214        |1049   |3        |20     |3         |2               |
|267341 |211        |327    |3        |20     |3         |80              |
|252753 |241        |556    |2        |20     |3         |50              |
|415978 |212        |336    |3        |16     |3         |90              |
|284221 |60         |964    |2        |1      |3         |93              |
|58210  |223        |200    |2        |18     |3         |67              |
|468306 |212        |170    |1        |20     |3         |119             |
+-------+---

In [32]:
df_fact_final.count() ## 1789439

1789439

## Incremental Update (Merge)

In [33]:
# Setup
dwh_location = "spark-warehouse/fact_treasure_hunt"

# Initialize table if it doesn't exist
if not os.path.exists(dwh_location):
    DeltaTable.create(spark) \
        .tableName("fact_treasure_hunt") \
        .addColumn("treasure_hunt_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
        .addColumn("user_sk", LongType(), nullable=False) \
        .addColumn("treasure_sk", IntegerType(), nullable=False) \
        .addColumn("date_SK", LongType(), nullable=False) \
        .addColumn("season_sk", IntegerType(), nullable=False) \
        .addColumn("hour_sk", IntegerType(), nullable=False) \
        .addColumn("weather_sk", LongType(), nullable=False) \
        .addColumn("duration_minutes", IntegerType(), nullable=False) \
        .property("delta.feature.identityColumns", "supported") \
        .execute()

# Load the Delta table
dt_fact_treasure_hunt = DeltaTable.forPath(spark, dwh_location)

# Perform the merge
dt_fact_treasure_hunt.alias("target").merge(
    source=df_fact_final.alias("source"),
    condition="target.user_sk = source.user_sk AND target.treasure_sk = source.treasure_sk AND target.date_sk = source.date_sk AND target.hour_sk = source.hour_sk"
).whenNotMatchedInsert(values={
    "user_sk": "source.user_sk",
    "treasure_sk": "source.treasure_sk",
    "date_sk": "source.date_sk",
    "season_sk": "source.season_sk",
    "hour_sk": "source.hour_sk",
    "weather_sk": "source.weather_sk",
    "duration_minutes": "source.duration_minutes"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Verify specific new data
We inserted a record with date `2023-09-12` and hour `10`. 
Let's check if it exists in the fact table.

In [37]:
dim_date.filter("full_date = '2023-09-12'").show() ## 1096

+-------+----------+---+-----+----+----------+------------+----------+
|date_sk| full_date|day|month|year|month_name|weekday_name|is_weekend|
+-------+----------+---+-----+----+----------+------------+----------+
|   1096|2023-09-12| 12|    9|2023| September|     Tuesday|     false|
+-------+----------+---+-----+----+----------+------------+----------+



In [38]:
spark.read.format("delta").load(dwh_location) \
    .filter(f"date_sk = {1096} AND hour_sk = 10") \
    .show(truncate=False)

+----------------+-------+-----------+-------+---------+-------+----------+----------------+
|treasure_hunt_sk|user_sk|treasure_sk|date_SK|season_sk|hour_sk|weather_sk|duration_minutes|
+----------------+-------+-----------+-------+---------+-------+----------+----------------+
|1151447         |59218  |182        |1096   |4        |10     |3         |79              |
|1805238         |3      |41         |1096   |4        |10     |3         |30              |
+----------------+-------+-----------+-------+---------+-------+----------+----------------+



## Cleaning up the environment (stop Spark and reset variables)

In [39]:
spark.stop()

In [40]:
%reset